In [18]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [21]:
params = {"command": "uv", "args": ["run", "current_date.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

In [22]:
mcp_tools

[Tool(name='get_current_date', title=None, description="Returns today's date", inputSchema={'properties': {}, 'title': 'get_current_dateArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'get_current_dateOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]

In [8]:
instructions = "You are a model which tells current date time"
request = "what is the date today"
model=  "gpt-4.1-mini"

In [15]:
from trace import Trace


async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="dateTimeTeller", instructions=instructions,  model=model, mcp_servers=[mcp_server])
    with trace("current_date"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Today's date is April 5, 2026.

In [ ]:
from openai import OpenAI
request = "what is the date today"
messages = [{"role": "user", "content": request}]
tools =[{"type":"mcp", "server_label": "mcp_server",}]
client = OpenAI()
response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{
        "type": "mcp",
        "server_label": "date_server",
        "server_url": "http://public.url/mcp",
        "require_approval": "never"
    }],
    input="what is the date today"
)

display(Markdown(response.choices[0].message.content))

APIStatusError: Error code: 424 - {'error': {'message': "Error retrieving tool list from MCP server: 'date_server'. Http status code: 424 (Failed Dependency)", 'type': 'external_connector_error', 'param': 'tools', 'code': 'http_error'}}